# 第 12 章 アンサンブル学習

弱い学習器を集めて強い学習器を作ります。多数決（バギング）と逐次的な重み付け（AdaBoost）を比べます。

対応する記事: [第 12 章 アンサンブル学習（Kotlin 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/kotlin/ch12.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch09.*
import ch12.*

## 切り株 1 本では解けないデータ

1 次元上に 3 つの領域が並んでいます。深さ 1 の木（切り株）は **線を 1 本しか引けない** ので、3 領域は分けられません。

In [2]:
val points = (1..8).map { listOf(it.toDouble(), 1.0) }
val labels = listOf(1, 1, -1, -1, -1, -1, 1, 1)

val stump = ch09.buildTree(points, labels, maxDepth = 1)
println(stump)
println("切り株 1 本の正解率 %.2f".format(treeAccuracy(stump, points, labels)))

Node(split=Split(feature=0, threshold=2.5), left=Leaf(label=1), right=Leaf(label=-1))


切り株 1 本の正解率 0.75


## バギングと AdaBoost を比べる

**同じ弱学習器を同じ本数使っても、束ね方で結果が分かれます。**

バギングが効かないのは、復元抽出でデータを揺らしても **どの標本でも「x = 2.5 で切る」のが最良** なので、10 本ともほぼ同じ木になるからです。同じ意見を 10 回聞いても結論は変わりません。

In [3]:
val forest = trainForest(points, labels, treeCount = 10, maxDepth = 1)
val boosted = trainAdaBoost(points, labels, rounds = 10, maxDepth = 1)

println("切り株 1 本            %.2f".format(treeAccuracy(stump, points, labels)))
println("バギング（10 本）      %.2f".format(accuracy(forest, points, labels)))
println("AdaBoost（10 本）      %.2f".format(accuracy(boosted, points, labels)))

切り株 1 本            0.75
バギング（10 本）      0.75
AdaBoost（10 本）      1.00


## AdaBoost が生む役割分担

**1 本目は左端、2 本目は右端に注目しました。** 1 本目が右端を外したので、その点の重みが上がり、2 本目が拾いに行ったのです。誰も指示していないのに役割分担が生まれます。

In [4]:
boosted.learners.take(4).forEachIndexed { index, learner ->
    println("ラウンド %d  発言権 %.4f".format(index + 1, learner.weight))
    println("          ${learner.tree}")
}

ラウンド 1  発言権 0.5493


          Node(split=Split(feature=0, threshold=2.5), left=Leaf(lab

el=1), right=Leaf(label=-1))
ラウンド 2  発言権 0.8047


          Node(split=Split(feature=0, threshold=6.5), left=Leaf(label=-1), right=Leaf(label=1))


ラウンド 3  発言権 0.6931


          Node(split=Split(feature=0, threshold=2.5), left=Leaf(label=1), right=Leaf(label=1))


ラウンド 4  発言権 0.7332


          Node(split=Split(feature=0, threshold=2.5), left=Leaf(label=1), right=Leaf(label=-1))


## 発言権の式

**誤り率 0.5 でちょうど 0 になります。** 当てずっぽうの意見は無視されます。0.5 より悪い学習器は「逆を言えば当たる」ので負の発言権になります。

In [5]:
println("%8s %10s".format("誤り率", "発言権"))
listOf(0.0, 0.05, 0.2, 0.4, 0.5, 0.7).forEach { error ->
    println("%8.2f %10.4f".format(error, learnerWeight(error)))
}

     誤り率        発言権
    0.00    11.5129
    0.05     1.4722
    0.20     0.6931


    0.40     0.2027
    0.50     0.0000
    0.70    -0.4236


## 試してみる: ラウンド数を変える

In [6]:
listOf(1, 2, 3, 5, 10).forEach { rounds ->
    val m = trainAdaBoost(points, labels, rounds = rounds, maxDepth = 1)
    println("%3d ラウンド  学習器 %2d 本  正解率 %.2f".format(rounds, m.learners.size,
            accuracy(m, points, labels)))
}

  1 ラウンド  学習器  1 本  正解率 0.75


  2 ラウンド  学習器  2 本  正解率 0.75


  3 ラウンド  学習器  3 本  正解率 1.00


  5 ラウンド  学習器  5 本  正解率 1.00


 10 ラウンド  学習器 10 本  正解率 1.00
